# Schedule RAG Agent — Colab Prototype

Two tools: `schedule_maker` (save a plan) and `get_schedule` (look up a day).
Backed by Pinecone (vector store) + Groq (LLM) via LangChain 1.x's `create_agent`.

Get free API keys before running:
- Groq: https://console.groq.com/keys
- Pinecone: https://app.pinecone.io (free tier includes a serverless index)

In [ ]:
!pip install -q langchain==1.3.14 langchain-groq==1.1.1 pinecone==7.3.0 sentence-transformers==5.1.1 python-dotenv==1.0.1

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Groq API key: ")
os.environ["PINECONE_API_KEY"] = getpass("Pinecone API key: ")
os.environ["PINECONE_INDEX_NAME"] = "schedule-agent"
os.environ["PINECONE_CLOUD"] = "aws"
os.environ["PINECONE_REGION"] = "us-east-1"
os.environ["GROQ_MODEL"] = "llama-3.3-70b-versatile"
os.environ["EMBEDDING_MODEL"] = "sentence-transformers/all-MiniLM-L6-v2"

## Set up the vector store and tools

This mirrors `app/tools.py` in the repo. Kept inline here so the notebook
runs standalone before you've pushed anything to GitHub.

In [ ]:
import uuid
from datetime import datetime

from langchain_core.tools import tool
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
PINECONE_INDEX_NAME = os.environ["PINECONE_INDEX_NAME"]
PINECONE_CLOUD = os.environ["PINECONE_CLOUD"]
PINECONE_REGION = os.environ["PINECONE_REGION"]
EMBEDDING_MODEL = os.environ["EMBEDDING_MODEL"]
EMBEDDING_DIMENSION = 384

pc = Pinecone(api_key=PINECONE_API_KEY)
model = SentenceTransformer(EMBEDDING_MODEL)

existing = [idx["name"] for idx in pc.list_indexes()]
if PINECONE_INDEX_NAME not in existing:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
index = pc.Index(PINECONE_INDEX_NAME)


def embed(text):
    return model.encode(text, normalize_embeddings=True).tolist()


def normalize_date(date_str):
    date_str = date_str.strip()
    for fmt in ("%Y-%m-%d", "%d-%m-%Y", "%d/%m/%Y", "%m/%d/%Y", "%B %d, %Y", "%d %B %Y"):
        try:
            return datetime.strptime(date_str, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return date_str


@tool
def schedule_maker(date: str, purpose: str) -> str:
    """Save a plan/commitment for a specific date (YYYY-MM-DD) with a purpose."""
    norm_date = normalize_date(date)
    vector_id = f"{norm_date}-{uuid.uuid4().hex[:8]}"
    index.upsert(vectors=[{
        "id": vector_id,
        "values": embed(purpose),
        "metadata": {"date": norm_date, "purpose": purpose},
    }])
    return f"Saved: on {norm_date} you have '{purpose}'."


@tool
def get_schedule(date: str) -> str:
    """Look up everything planned for a specific date (YYYY-MM-DD)."""
    norm_date = normalize_date(date)
    result = index.query(
        vector=embed(norm_date),
        top_k=20,
        filter={"date": {"$eq": norm_date}},
        include_metadata=True,
    )
    matches = result.get("matches", [])
    if not matches:
        return f"No plans found for {norm_date}. That day looks free."
    items = [m["metadata"]["purpose"] for m in matches]
    return f"On {norm_date} you have: {'; '.join(items)}."

## Build the agent

In [ ]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

llm = ChatGroq(model=os.environ["GROQ_MODEL"], temperature=0)
today = datetime.now().strftime("%Y-%m-%d (%A)")

system_prompt = (
    f"You are a scheduling assistant. Today's date is {today}.\n\n"
    "Rules:\n"
    "1. Always resolve relative dates ('tomorrow', 'next Friday', 'this weekend') "
    "into an exact YYYY-MM-DD date before calling any tool.\n"
    "2. Use `schedule_maker` whenever the person tells you about a plan to save.\n"
    "3. Use `get_schedule` whenever the person asks what they're doing or whether "
    "something conflicts with an existing plan.\n"
    "4. For conflict-check questions, call `get_schedule` first, then reason about "
    "whether it conflicts, and explain clearly.\n"
    "5. Never invent schedule entries — only report what the tool returns."
)

graph = create_agent(model=llm, tools=[schedule_maker, get_schedule], system_prompt=system_prompt)


def ask(question):
    result = graph.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content

## Try it out

In [ ]:
print(ask("I have a team offsite on 2026-08-20"))
print(ask("On 2026-08-21 I have a dentist appointment at 3pm"))

In [ ]:
print(ask("I want to go watch the new Spiderman movie on 2026-08-20, do I have any work that day?"))

In [ ]:
print(ask("What about 2026-08-22, am I free?"))

## Next steps

Once this behaves the way you want:
1. Copy your working `GROQ_API_KEY` / `PINECONE_API_KEY` values into a local `.env` file (never commit this).
2. Push the `schedule-rag-agent` repo to GitHub (see `README.md`).
3. Deploy `app/server.py` to Render (see `README.md`).